# BIU DS23 · Module 4 · The Pipeline Assignment · Starter Notebook
**Due August 30. Read the assignment document first, then work here.**

This notebook is scaffolding, not a solution. Your job is to build one leakage-safe
Pipeline, from raw data to model, and to JUSTIFY every decision. Remember the rubric:
**70% of the grade is on your reasoning, not on the score.** A small or even negative
lift with excellent justification beats a high score you cannot explain.

**Two tracks (pick one, see the assignment document):**
1. **Olist marketplace.** Beat the frozen benchmark you generated by running
   `DS23_Module4_Benchmark.ipynb` (it writes `module4_benchmark.json` to your Drive).
2. **Your own data.** The same process on your capstone dataset. You build your own
   minimal baseline first, then measure your lift against it.

**The one rule that never bends:** every step that learns from the data (imputation,
scaling, encoding, resampling) lives INSIDE the Pipeline, so it is fit on the training
fold only. Anything else is leakage, and leakage caps your grade.

## 0 · Setup

In [2]:
!pip install -q scikit-learn pandas numpy imbalanced-learn category_encoders

import numpy as np, pandas as pd, json

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", None)

from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/שעורי בית/"


Mounted at /content/drive


## 1 · Load the raw tables (Olist track)
For the Olist track, load the raw tables. For your own-data track, load your dataset
instead and skip to section 3.

In [3]:
items    = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
reviews  = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
orders   = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
print("tables loaded.")


tables loaded.


## 2 · The frozen benchmark
You generated this by running `DS23_Module4_Benchmark.ipynb`. Load the locked number.
Your pipeline will be measured against it. Do NOT recompute it here.

In [4]:
with open(DATA_PATH + "module4_benchmark.json") as f:
    bench = json.load(f)
print("frozen benchmark roc_auc:", bench["roc_auc"])
print("protocol:", bench["cv"], "| metric:", bench["metric"])
# If this cell errors, run DS23_Module4_Benchmark.ipynb first to create the JSON.


frozen benchmark roc_auc: 0.5586
protocol: StratifiedKFold(5, shuffle=True, random_state=42) | metric: roc_auc


## 3 · Build the modeling table
The target is a negative review (score 1 or 2). This definition is fixed for everyone,
so results are comparable. Keep the SAME row set and the SAME evaluation protocol as the
benchmark, or the comparison is not fair.

In [5]:
base = (items
        .merge(products, on="product_id", how="left")
        .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
        .merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id", how="left"))
base = base.dropna(subset=["review_score"]).copy()
base["neg_review"] = (base["review_score"] <= 2).astype(int)
base["order_purchase_timestamp"] = pd.to_datetime(base["order_purchase_timestamp"])
base = base.sort_values("order_purchase_timestamp").reset_index(drop=True)
model_df = base.dropna(subset=["product_category_name"]).copy()
y = model_df["neg_review"]
print("modeling table:", model_df.shape, "| negative rate:", round(y.mean(), 4))


modeling table: (110774, 18) | negative rate: 0.1603


## 4 · The frozen evaluation protocol
Use these exact objects for every comparison. Same cv, same metric as the benchmark.

In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)   # identical to the benchmark
METRIC = "roc_auc"

def evaluate(pipe, X):
    return cross_val_score(pipe, X, y, cv=cv, scoring=METRIC).mean()


---
# Your work starts here
Each section has a TODO. Fill it, and record your decision and reasoning in the
justification template (section 2 of the assignment document). One sentence of code is
not enough; the WHY is what is graded.

## 5 · Profile and handle missing values
Profile the gaps first, classify them (MCAR / MAR / MNAR), then choose an imputation
strategy. It must live inside the Pipeline.

In [7]:
# TODO 5: profile missingness - counts and percentages

missing_profile = pd.DataFrame({
    "missing_count": model_df.isna().sum(),
    "missing_pct": model_df.isna().mean() * 100
})

missing_profile = (
    missing_profile[missing_profile["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)

missing_profile.round(2)

,missing_count,missing_pct
product_weight_g,1,0.0
product_length_cm,1,0.0
product_height_cm,1,0.0
product_width_cm,1,0.0


In [8]:
# TODO 5: inspect the missingness pattern

dimension_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_rows = model_df.loc[
    model_df[dimension_cols].isna().any(axis=1),
    ["product_id", "product_category_name"] + dimension_cols
]

print("Rows with at least one missing product dimension:", len(missing_rows))
display(missing_rows)

print("\nMissingness patterns:")
display(
    model_df[dimension_cols]
    .isna()
    .value_counts()
    .reset_index(name="count")
)

Rows with at least one missing product dimension: 1


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1160,09ff539a621711667c43eba6a3bd8466,bebes,NaN,NaN,NaN,NaN



Missingness patterns:


,product_weight_g,product_length_cm,product_height_cm,product_width_cm,count
0,False,False,False,False,110773
1,True,True,True,True,1


In [ ]:
# Missingness classification:
# The four product-dimension variables are missing together in the same single row.
# This co-missingness pattern suggests a shared measurement or data-collection process,
# so MAR is the most plausible classification.
# However, only one row is affected, so the evidence for the mechanism is limited.

In [9]:
# TODO 5: compare imputation strategies using the frozen CV protocol

numeric_features = [
    "price",
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

categorical_features = ["product_category_name"]

X_missing = model_df[numeric_features + categorical_features].copy()

def build_imputation_pipeline(strategy):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy=strategy)),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ])

    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ])

median_pipe = build_imputation_pipeline("median")
mean_pipe = build_imputation_pipeline("mean")

median_score = evaluate(median_pipe, X_missing)
mean_score = evaluate(mean_pipe, X_missing)

print("Median imputation ROC-AUC:", round(median_score, 4))
print("Mean imputation ROC-AUC:  ", round(mean_score, 4))
print("Difference:", round(median_score - mean_score, 6))

Median imputation ROC-AUC: 0.5586
Mean imputation ROC-AUC:   0.5588
Difference: -0.000148


In [ ]:
# Decision: use median imputation for the numeric product dimensions.
#
# Only one modeling row has missing product dimensions, and all four
# dimensions are missing together. This co-missingness pattern is most
# consistent with MAR, although the evidence is limited because only one
# row is affected.
#
# I compared median and mean imputation using the frozen 5-fold
# cross-validation protocol. Mean imputation achieved ROC-AUC 0.5588,
# while median imputation achieved 0.5586. The difference was only 0.000148,
# which I consider practically negligible.
#
# I therefore kept median imputation because it is less sensitive to
# extreme values and provides a simple, robust treatment without chasing
# a negligible CV difference.

## 6 · Outliers
Detect outliers (IQR, Z-score, or a multivariate method), then decide: remove, clip,
transform, or keep. Justify statistically AND in business terms.

In [10]:
# TODO 6: detect outliers using the IQR rule

outlier_features = [
    "price",
    "freight_value",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

outlier_summary = []

for col in outlier_features:
    s = model_df[col].dropna()

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (s < lower_bound) | (s > upper_bound)

    outlier_summary.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_mask.sum(),
        "outlier_pct": outlier_mask.mean() * 100,
        "min": s.min(),
        "max": s.max()
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(
    outlier_summary.round({
        "Q1": 2,
        "Q3": 2,
        "IQR": 2,
        "lower_bound": 2,
        "upper_bound": 2,
        "outlier_pct": 2,
        "min": 2,
        "max": 2
    })
)

,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_pct,min,max
0,price,39.90,134.90,95.00,-102.60,277.40,8292,7.49,0.85,6735.00
1,freight_value,13.08,21.17,8.09,0.94,33.31,11954,10.79,0.00,409.68
2,product_weight_g,300.00,1800.00,1500.00,-1950.00,4050.00,15588,14.07,0.00,40425.00
3,product_length_cm,18.00,38.00,20.00,-12.00,68.00,3566,3.22,7.00,105.00
4,product_height_cm,8.00,20.00,12.00,-10.00,38.00,7562,6.83,2.00,105.00
5,product_width_cm,15.00,30.00,15.00,-7.50,52.50,2531,2.28,6.00,118.00


In [11]:
# TODO 6: inspect skewness and extreme quantiles before choosing a treatment

skew_summary = []

for col in outlier_features:
    s = model_df[col].dropna()

    skew_summary.append({
        "feature": col,
        "skewness": s.skew(),
        "median": s.median(),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "p99_9": s.quantile(0.999),
        "max": s.max()
    })

skew_summary = pd.DataFrame(skew_summary)

display(
    skew_summary.round({
        "skewness": 2,
        "median": 2,
        "p95": 2,
        "p99": 2,
        "p99_9": 2,
        "max": 2
    })
)

,feature,skewness,median,p95,p99,p99_9,max
0,price,7.63,74.90,349.90,889.00,2094.52,6735.00
1,freight_value,5.65,16.29,45.19,84.38,175.58,409.68
2,product_weight_g,3.59,700.00,9750.00,18225.00,30000.00,40425.00
3,product_length_cm,1.76,25.00,62.00,95.00,105.00,105.00
4,product_height_cm,2.26,13.00,45.00,65.00,105.00,105.00
5,product_width_cm,1.72,20.00,45.00,62.00,92.00,118.00


In [12]:
# TODO 6: compare keeping the right-skewed variables as-is
# versus applying a log1p transformation inside the Pipeline.

from sklearn.preprocessing import FunctionTransformer

log_features = [
    "price",
    "freight_value",
    "product_weight_g"
]

regular_features = [
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

categorical_features = ["product_category_name"]

X_outliers = model_df[
    log_features + regular_features + categorical_features
].copy()


# Alternative 1: keep all numeric variables in their original scale
raw_numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

raw_preprocess = ColumnTransformer([
    ("num", raw_numeric_pipe, log_features + regular_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

raw_outlier_pipe = Pipeline([
    ("preprocess", raw_preprocess),
    ("model", LogisticRegression(max_iter=1000))
])


# Alternative 2: log-transform only the strongly right-skewed variables
log_numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

regular_numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

log_preprocess = ColumnTransformer([
    ("log_num", log_numeric_pipe, log_features),
    ("regular_num", regular_numeric_pipe, regular_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

log_outlier_pipe = Pipeline([
    ("preprocess", log_preprocess),
    ("model", LogisticRegression(max_iter=1000))
])


raw_score = evaluate(raw_outlier_pipe, X_outliers)
log_score = evaluate(log_outlier_pipe, X_outliers)

print("Keep raw ROC-AUC:", round(raw_score, 4))
print("Log transform ROC-AUC:", round(log_score, 4))
print("Difference:", round(log_score - raw_score, 6))

Keep raw ROC-AUC: 0.5586
Log transform ROC-AUC: 0.559
Difference: 0.000354


In [ ]:
# Decision: keep all rows and apply log1p to price, freight_value,
# and product_weight_g inside the Pipeline.
#
# IQR profiling identified many statistically extreme observations,
# especially for product weight, freight value, and price. These variables
# were also strongly right-skewed (skewness = 3.59, 5.65, and 7.63).
#
# I did not remove the outliers because extreme prices, freight costs,
# and product weights can represent valid marketplace transactions rather
# than data errors. Removing them would also change the frozen benchmark
# row set.
#
# I compared keeping the variables in their original scale with applying
# log1p inside the Pipeline. The raw version achieved ROC-AUC 0.5586,
# while the log-transformed version achieved 0.5590, an improvement of
# 0.000354.
#
# The gain is small, so I do not interpret it as a major performance
# improvement. I selected the log transformation because it has both a
# statistical justification (strong right skew) and a business advantage:
# it reduces the influence of extreme values without deleting valid cases.

## 7 · Assemble the leakage-safe Pipeline
Build a ColumnTransformer (numeric route + categorical route) and wrap it with a model.
Everything that learns from data goes inside. This skeleton is a starting point; change
the strategies to the ones YOU justified above.

In [13]:
# TODO 7A: compare scaling vs. no scaling using the frozen CV protocol

from sklearn.preprocessing import FunctionTransformer

log_cols = [
    "price",
    "freight_value",
    "product_weight_g"
]

regular_num_cols = [
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

cat_col = "product_category_name"

feature_cols = log_cols + regular_num_cols + [cat_col]
X_pipe = model_df[feature_cols].copy()


def build_scaling_pipeline(use_scaling=True):

    log_steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))
    ]

    regular_steps = [
        ("impute", SimpleImputer(strategy="median"))
    ]

    if use_scaling:
        log_steps.append(("scale", StandardScaler()))
        regular_steps.append(("scale", StandardScaler()))

    log_numeric = Pipeline(log_steps)
    regular_numeric = Pipeline(regular_steps)

    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("log_num", log_numeric, log_cols),
        ("regular_num", regular_numeric, regular_num_cols),
        ("cat", categorical, [cat_col])
    ])

    return Pipeline([
        ("pre", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ])


scaled_pipe = build_scaling_pipeline(use_scaling=True)
unscaled_pipe = build_scaling_pipeline(use_scaling=False)

scaled_score = evaluate(scaled_pipe, X_pipe)
unscaled_score = evaluate(unscaled_pipe, X_pipe)

print("With StandardScaler ROC-AUC:", round(scaled_score, 4))
print("Without scaling ROC-AUC:    ", round(unscaled_score, 4))
print("Difference:", round(scaled_score - unscaled_score, 6))

With StandardScaler ROC-AUC: 0.559
Without scaling ROC-AUC:     0.5588
Difference: 0.000138


In [14]:
# TODO 7B: compare OneHotEncoder with an ordinal encoding alternative

from sklearn.preprocessing import OrdinalEncoder

def build_encoding_pipeline(encoding="onehot"):

    log_numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler())
    ])

    regular_numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ])

    if encoding == "onehot":
        categorical = Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore"))
        ])

    else:
        categorical = Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encode", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ])

    preprocessor = ColumnTransformer([
        ("log_num", log_numeric, log_cols),
        ("regular_num", regular_numeric, regular_num_cols),
        ("cat", categorical, [cat_col])
    ])

    return Pipeline([
        ("pre", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ])


onehot_pipe = build_encoding_pipeline("onehot")
ordinal_pipe = build_encoding_pipeline("ordinal")

onehot_score = evaluate(onehot_pipe, X_pipe)
ordinal_score = evaluate(ordinal_pipe, X_pipe)

print("One-Hot ROC-AUC:", round(onehot_score, 4))
print("Ordinal ROC-AUC:", round(ordinal_score, 4))
print("Difference:", round(onehot_score - ordinal_score, 6))

One-Hot ROC-AUC: 0.559
Ordinal ROC-AUC: 0.5243
Difference: 0.0347


In [15]:
# TODO 7: final leakage-safe preprocessing pipeline

from sklearn.preprocessing import FunctionTransformer

# Strongly right-skewed numeric variables:
log_cols = [
    "price",
    "freight_value",
    "product_weight_g"
]

# Numeric variables kept on their original scale:
regular_num_cols = [
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

cat_col = "product_category_name"

# Explicit feature set - avoid automatically including unrelated/leaky columns
feature_cols = log_cols + regular_num_cols + [cat_col]
X = model_df[feature_cols].copy()


# Numeric route for strongly right-skewed variables
log_numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler())
])


# Numeric route for the remaining dimensions
regular_numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])


# Categorical route
categorical = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


# Combine preprocessing routes
pre = ColumnTransformer([
    ("log_num", log_numeric, log_cols),
    ("regular_num", regular_numeric, regular_num_cols),
    ("cat", categorical, [cat_col])
])


# Final leakage-safe Pipeline
pipe = Pipeline([
    ("pre", pre),
    ("model", LogisticRegression(max_iter=1000))
])

print(pipe)


Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('log_num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log',
                                                                   FunctionTransformer(feature_names_out='one-to-one',
                                                                                       func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['price', 'freight_value',
                                                   'product_weight_g']),
                                                 ('regular_num',
                                                  Pip

## 8 · Handle class imbalance
Only about one in six reviews is negative, and those are the ones that matter. Choose a
strategy (class_weight, resampling via imblearn Pipeline, or threshold tuning) and the
right metric. accuracy is not the right metric here; explain why.

In [16]:
# TODO 8A: inspect class imbalance and show why accuracy is misleading

class_summary = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "percentage": (y.value_counts(normalize=True).sort_index() * 100)
})

class_summary.index = ["non-negative review (0)", "negative review (1)"]

display(class_summary.round(2))

majority_accuracy = (y == 0).mean()

print("Negative review rate:", round(y.mean(), 4))
print("Naive majority-class accuracy:", round(majority_accuracy, 4))


,count,percentage
non-negative review (0),93018,83.97
negative review (1),17756,16.03


Negative review rate: 0.1603
Naive majority-class accuracy: 0.8397


In [17]:
# TODO 8B: get leakage-safe out-of-fold probabilities
# and inspect performance at the default threshold of 0.5

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

oof_prob = cross_val_predict(
    pipe,
    X,
    y,
    cv=cv,
    method="predict_proba"
)[:, 1]

default_pred = (oof_prob >= 0.5).astype(int)

print("ROC-AUC:", round(roc_auc_score(y, oof_prob), 4))
print("Precision at threshold 0.5:", round(precision_score(y, default_pred, zero_division=0), 4))
print("Recall at threshold 0.5:   ", round(recall_score(y, default_pred), 4))
print("F1 at threshold 0.5:       ", round(f1_score(y, default_pred), 4))

ROC-AUC: 0.5587
Precision at threshold 0.5: 0.0
Recall at threshold 0.5:    0.0
F1 at threshold 0.5:        0.0


In [18]:
# TODO 8C: inspect the distribution of predicted probabilities

prob_summary = pd.Series(oof_prob).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

display(prob_summary)

print("Maximum predicted probability:", round(oof_prob.max(), 4))
print("Share predicted >= 0.5:", round((oof_prob >= 0.5).mean(), 4))

,0
count,110774.000000
mean,0.160292
std,0.031289
min,0.059916
50%,0.156612
75%,0.180327
90%,0.196167
95%,0.209083
99%,0.267337
max,0.392852


Maximum predicted probability: 0.3929
Share predicted >= 0.5: 0.0


In [19]:
# TODO 8D: compare candidate decision thresholds

thresholds = [0.10, 0.12, 0.14, 0.16, 0.18, 0.20, 0.25, 0.30]

threshold_results = []

for threshold in thresholds:
    pred = (oof_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "predicted_positive_pct": pred.mean() * 100
    })

threshold_results = pd.DataFrame(threshold_results)

display(
    threshold_results.round({
        "precision": 4,
        "recall": 4,
        "f1": 4,
        "predicted_positive_pct": 2
    })
)

,threshold,precision,recall,f1,predicted_positive_pct
0,0.10,0.1610,0.9891,0.2769,98.48
1,0.12,0.1624,0.9586,0.2777,94.63
2,0.14,0.1716,0.7795,0.2812,72.82
3,0.16,0.1851,0.5287,0.2743,45.77
4,0.18,0.1999,0.3162,0.2449,25.35
5,0.20,0.2263,0.1127,0.1505,7.98
6,0.25,0.2743,0.0269,0.0489,1.57
7,0.30,0.3125,0.0034,0.0067,0.17


In [20]:
# TODO 8: final imbalance decision

FINAL_THRESHOLD = 0.14

final_pred = (oof_prob >= FINAL_THRESHOLD).astype(int)

print("Chosen threshold:", FINAL_THRESHOLD)
print("ROC-AUC:", round(roc_auc_score(y, oof_prob), 4))
print("Precision:", round(precision_score(y, final_pred), 4))
print("Recall:", round(recall_score(y, final_pred), 4))
print("F1:", round(f1_score(y, final_pred), 4))
print("Predicted positive rate:", round(final_pred.mean(), 4))

# Decision:
# The target is imbalanced: only 16.03% of observations are negative reviews.
# Accuracy is therefore misleading; predicting the majority class for every row
# would already achieve about 83.97% accuracy while detecting zero negative reviews.
#
# ROC-AUC is kept as the primary evaluation metric because it is threshold-independent
# and is also the frozen benchmark metric.
#
# For the operational classification threshold, I used threshold tuning rather than
# changing the model with class weights or resampling.
#
# At the default threshold of 0.5, the model predicted no positive cases:
# precision = 0, recall = 0, and F1 = 0.
#
# Among the tested thresholds, 0.14 achieved the highest F1 (0.2812) while retaining
# high recall (0.7795). The trade-off is low precision (0.1716) and a high predicted
# positive rate (72.82%), which reflects the weak separation available from the
# current feature set.

Chosen threshold: 0.14
ROC-AUC: 0.5587
Precision: 0.1716
Recall: 0.7795
F1: 0.2812
Predicted positive rate: 0.7282


## 9 · Measure your lift against the benchmark
Same cv, same metric. Report your score, the benchmark, and the lift, honestly.

In [21]:
# Measure lift against the frozen benchmark

X_final = model_df[feature_cols].copy()

my_score = evaluate(pipe, X_final)
lift = my_score - bench["roc_auc"]

print(f"benchmark : {bench['roc_auc']:.4f}")
print(f"my score  : {my_score:.4f}")
print(f"my lift   : {lift:+.4f}")


benchmark : 0.5586
my score  : 0.5590
my lift   : +0.0004


In [ ]:
# Interpretation:
# The final pipeline achieved ROC-AUC 0.5590 compared with the frozen
# benchmark score of 0.5586, for a lift of +0.0004.
#
# This is a small positive improvement rather than a substantial gain.
# It suggests that the preprocessing decisions improved the data representation
# slightly, but the available feature set still contains only weak predictive
# signal for negative reviews.
#
# The result is still meaningful because the comparison used the same model,
# same cross-validation protocol, same seed, same metric, and the same row set.

## 10 · Own-data track only: your own baseline
If you chose the own-data track, you have no external benchmark. Build a minimal
baseline (a simple leakage-safe pipeline on your raw features), lock its score, and
measure the lift of your cleaned and engineered pipeline against it. Same idea, same
discipline.

In [ ]:
# TODO 10: Not applicable.
# I used the Olist benchmark track, so the frozen external benchmark
# was used instead of building a separate own-data baseline.

---
## Before you submit
- Every learning step is inside the Pipeline (no leakage). Check twice.
- The justification template is filled for every decision: what, why, alternative
  rejected, evidence.
- Your lift is reported honestly, whatever it is.
- The notebook runs top to bottom without errors.

Good luck. The reasoning is the point.